# **Preprocesamiento de datos**

In [25]:
import pandas as pd
import numpy as np
import re

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter, MaxNLocator

In [26]:
dataset = pd.read_csv("data/raw.csv")
raw_dataset = dataset.copy()

# Nuevas columnas

### Turbo

Buscar "T", "turbo" en la columna "Motor". También podríamos buscar en la descripción, versión o el título (ahora es solo en "Motor")

In [27]:
def is_turbo(row):
    motor = str(row.get('Motor', ''))
    version = str(row.get('Versión', ''))
    combined = f"{motor} {version}"
    # Look for 'turbo' or typical turbo patterns
    if re.search(r'turbo', combined, re.IGNORECASE):
        return 1
    if re.search(r'(?:\d[\.,]?\d?\s*[tT]|TSI|TFSI|TURBO)', combined, re.IGNORECASE):
        return 1
    return 0

dataset['Turbo'] = dataset.apply(is_turbo, axis=1)

### Caballos de fuerza (cv)


In [28]:
def extract_cv(text):
    if not isinstance(text, str):
        return np.nan
    match = re.search(r'(\d+)\s*[cC][vV]\b', text)
    if match:
        return int(match.group(1))
    return np.nan

dataset['cv'] = dataset['Versión'].apply(extract_cv)

## Tracción

In [29]:
def extract_traccion(text):
    if not isinstance(text, str):
        return np.nan
    if re.search(r'4[xX]4|4wd|awd', text, re.IGNORECASE):
        return '4x4'
    if re.search(r'4[xX]2|2wd|fwd', text, re.IGNORECASE):
        return '4x2'
    return np.nan

dataset['Tracción'] = dataset['Versión'].apply(extract_traccion)

# Transformación de columnas

## **Motor**

### Buscar el tamaño del motor en "Versión" si no está en motor

Ejemplo: Fila 86 del archivo @data/raw.csv 
- `Motor` es **TURBOALIMENTADO** y `Versión` es **2.0 T ALLSPACE COMFORTLINE AT L17**

In [30]:
def extract_engine_size(version):
    if not isinstance(version, str):
        return None
    match = re.match(r'(\d\.\d)', version)
    if match:
        return match.group(1)
    return None

def fix_motor(row):
    motor = row['Motor']
    version = row['Versión']
    try:
        float(motor)
        is_number = True
    except (ValueError, TypeError):
        is_number = False
    if not is_number:
        engine_size = extract_engine_size(version)
        if engine_size:
            return engine_size
    return motor

dataset['Motor'] = dataset.apply(fix_motor, axis=1)

print(dataset.iloc[84])

idx                                                                       84
Marca                                                             Volkswagen
Modelo                                                                Tiguan
Año                                                                   2018.0
Versión                                    2.0 T ALLSPACE COMFORTLINE AT L17
Color                                                                 BLANCA
Tipo de combustible                                                    Nafta
Puertas                                                                  5.0
Transmisión                                                       Automática
Motor                                                                    2.0
Tipo de carrocería                                                       SUV
Con cámara de retroceso                                                  NaN
Kilómetros                                                           89000.0

## **Versión**

### Borrar el cilindraje.

Ejemplo: borrar 2.4 de "2.4 Sport At"

In [31]:
def remove_cilinder_size(version):
    if not isinstance(version, str):
        return version
    # Remove patterns like "1.0", "1,0", "2.0", "2,0", "1.6", "1,6", "2.4", "2,4", "1.5", "1,5", etc.
    # Allow optional "L" or "l" after the number, and optional space
    # Only match at the start or after a space, and only if followed by space or end or non-word char
    pattern = r'(^|\s)(\d[\.,]\d)\s*[Ll]?(?=\s|$|\W)'
    version = re.sub(pattern, ' ', version)
    version = re.sub(r'\b\d[\.,]?\d?\s*[tT]\b', ' ', version)
    version = re.sub(r'\b(TSI|TFSI|TURBO)\b', ' ', version, flags=re.IGNORECASE)
    # Remove extra spaces
    version = re.sub(' +', ' ', version).strip()
    return version

dataset['Versión'] = dataset['Versión'].apply(remove_cilinder_size)

### Borrar "at" y "mt"
(Manual/Automatic transmission) 

Ya existe una columna para eso, pero algunos datos tienen la cantidad de marchas que tiene la transmisión (6at o AT8)

In [32]:
def remove_transmission(text):
    if not isinstance(text, str):
        return text
    # Remove patterns like "6at", "at4", "8mt", etc.
    text = re.sub(r'\b\d*\s*(at|mt|dct|cvt)\d*\b', '', text, flags=re.IGNORECASE)
    # Remove standalone transmission types and abbreviations
    text = re.sub(r'\b(at|mt|dct|cvt|automática|manual)\b', '', text, flags=re.IGNORECASE)
    return re.sub(' +', ' ', text).strip()

dataset['Versión'] = dataset['Versión'].apply(remove_transmission)

### Borrar los caballos de fuerza (cv o hp)

In [33]:
def remove_horsepower(text):
    if not isinstance(text, str):
        return text
    return re.sub(r'\b\d+\s*(cv|hp)\b', '', text, flags=re.IGNORECASE).replace('  ', ' ').strip()

# Apply to your dataset
dataset['Versión'] = dataset['Versión'].apply(remove_horsepower)

### Borrar la tracción

4x2 o 4x4

In [34]:
def remove_traction(text):
    if not isinstance(text, str):
        return text
    # Remove '4x4', '4x2', 'awd', '4wd', '2wd', 'fwd' (case-insensitive, x can be X)
    text = re.sub(r'\b4[xX][24]\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bawd\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b4wd\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b2wd\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bfwd\b', '', text, flags=re.IGNORECASE)
    return re.sub(' +', ' ', text).strip()

dataset['Versión'] = dataset['Versión'].apply(remove_traction)

### Borrar otros datos

"Nafta", el modelo del auto o la marca, "SUV"

In [35]:
def remove_other_data(row):
    version = str(row.get('Versión', ''))
    marca = str(row.get('Marca', '')).lower()
    modelo = str(row.get('Modelo', '')).lower()
    combustible = str(row.get('Tipo de combustible', '')).lower()
    carroceria = str(row.get('Tipo de carrocería', '')).lower()
    
    version_lower = version.lower()
    
    # Remove fuel type
    if combustible and combustible != 'nan':
        version_lower = re.sub(rf'\b{re.escape(combustible)}\b', '', version_lower)
    
    # Remove brand
    if marca and marca != 'nan':
        version_lower = re.sub(rf'\b{re.escape(marca)}\b', '', version_lower)
    
    # Remove model
    if modelo and modelo != 'nan':
        version_lower = re.sub(rf'\b{re.escape(modelo)}\b', '', version_lower)
    
    # Remove body type
    if carroceria and carroceria != 'nan':
        version_lower = re.sub(rf'\b{re.escape(carroceria)}\b', '', version_lower)
    
    # Remove common fuel types that might not be in the combustible column
    version_lower = re.sub(r'\b(nafta|gasolina|diesel|gnc|gas|híbrido|eléctrico)\b', '', version_lower)
    
    # Remove common body types that might not be in the carroceria column
    version_lower = re.sub(r'\b(suv|sedan|hatchback|pickup|coupe|convertible)\b', '', version_lower)
    
    # Clean up extra spaces
    version_cleaned = re.sub(r'\s+', ' ', version_lower).strip()
    
    return version_cleaned

dataset['Versión'] = dataset.apply(remove_other_data, axis=1)

## Color

In [14]:
dataset['Color'].value_counts()

Color
Gris                             5468
Blanco                           4809
Negro                            2825
Plateado                         1427
Azul                             1121
                                 ... 
GRIS SILVERSTONE                    1
BLANCO BANCHISA BICOLOR NEGRO       1
SKYSCRAPER GRAU METALLIC            1
GRANITE CRYSTA BC                   1
GRIS PLATINO                        1
Name: count, Length: 70, dtype: int64

In [26]:
import pandas as pd
import difflib
import unicodedata

def limpiar_columna_color(df, columna='Color'):
    # Lista fija de categorías estándar
    colores_base = [
        "blanco", "negro", "gris", "plateado", "azul", "rojo", "verde",
        "dorado", "amarillo", "naranja", "marron", "violeta", "rosa", "otro"
    ]

    # Función para normalizar texto
    def normalizar(texto):
        if not isinstance(texto, str):
            return ''
        texto = texto.lower().strip()
        texto = ''.join(
            c for c in unicodedata.normalize('NFKD', texto)
            if not unicodedata.combining(c)
        )
        return texto

    # Función para asignar el color más parecido
    def asignar_color_mas_parecido(color):
        color_norm = normalizar(color)
        match = difflib.get_close_matches(color_norm, colores_base, n=1, cutoff=0.6)
        return match[0] if match else 'otro'

    # Crear columna nueva en el DataFrame original
    df[columna + '_limpio'] = df[columna].apply(asignar_color_mas_parecido)
    return df

dataset = limpiar_columna_color(dataset, columna='Color')
dataset['Color_limpio'].value_counts()


Color_limpio
gris        5469
blanco      4820
negro       2834
plateado    1436
azul        1121
rojo        1118
otro         734
marron       257
dorado       191
verde        135
naranja       73
violeta       33
amarillo      25
rosa           8
Name: count, dtype: int64

In [24]:
dataset['Color_limpio'].value_counts()

Color_limpio
gris plata                5521
blanca                    4825
negra                     2832
plata                     1437
rojo                      1132
azul                      1123
Otro                       389
marron                     282
dorado                     221
verde                      138
beige                      125
celeste                     93
naranja                     73
violeta                     33
gris selenium                4
cafe                         3
steel_blue                   3
mineralweiss metallic        3
blanco nacre tricapa         3
granite crystal bc           2
carbon black                 2
summit white                 2
cobre                        1
moundaz                      1
beige techo negro            1
rojo sunset metalizado       1
black meet kettle            1
noir perla nera              1
champaing                    1
prata bari+tet vulc          1
Name: count, dtype: int64

# Guardar en CSV

In [ ]:

dataset.to_csv("data/clean.csv", index=False)